In [79]:
import pandas as pd 

data = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
test_ids = test["PassengerId"]

In [80]:
data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [81]:
def clean(data):
    
    cols = ["SibSp", "Parch", "Fare", "Age"]
    classes = [1, 2, 3]

    for col in cols:
        for clas in classes:
            median = data.loc[data["Pclass"] == clas, col].median()

            data.loc[
                (data["Pclass"] == clas) & (data[col].isna()),
                col
            ] = median

    data["Embarked"] = data["Embarked"].fillna("U")

    return data

data = clean(data)
test = clean(test)

In [82]:
# feature engineer 
for df in [data,test]:
    df["family_size"] = df["Parch"] + df["SibSp"] + 1
    df["is_child"] = (df["Age"] < 14).astype(int)
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
    

In [83]:
data = data.drop(["Cabin","Name", "PassengerId"],axis=1)
test = test.drop(["Cabin","Name", "PassengerId"],axis=1)


In [84]:
print (data["Ticket"].nunique())

681


In [85]:
print(len(data))

891


In [86]:
ticket_counts = pd.concat([data["Ticket"], test["Ticket"]]).value_counts()
for df in [data, test]:
    df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)

In [87]:
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,family_size,is_child,is_alone,Title,TicketGroupSize
0,0,3,male,22.0,1,0,A/5 21171,7.2500,S,2,0,0,Mr,1
1,1,1,female,38.0,1,0,PC 17599,71.2833,C,2,0,0,Mrs,2
2,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,S,1,0,1,Miss,1
3,1,1,female,35.0,1,0,113803,53.1000,S,2,0,0,Mrs,2
4,0,3,male,35.0,0,0,373450,8.0500,S,1,0,1,Mr,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,211536,13.0000,S,1,0,1,Rev,1
887,1,1,female,19.0,0,0,112053,30.0000,S,1,0,1,Miss,1
888,0,3,female,24.0,1,2,W./C. 6607,23.4500,S,4,0,0,Miss,4
889,1,1,male,26.0,0,0,111369,30.0000,C,1,0,1,Mr,1


In [88]:
data = data.drop("Ticket" , axis = 1)

In [89]:
agemean = data["Age"].mean()
agestd = data["Age"].std()
faremean = data["Fare"].mean()
farestd = data["Fare"].std()

data["Age"] = (data["Age"]-agemean)/agestd
test["Age"] = (test["Age"] - agemean) / agestd

data["Fare"] = (data["Fare"]-faremean)/farestd
test["Fare"] = (test["Fare"] - faremean) / farestd

In [90]:
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,family_size,is_child,is_alone,Title,TicketGroupSize
0,0,3,male,-0.533534,1,0,-0.502163,S,2,0,0,Mr,1
1,1,1,female,0.674512,1,0,0.786404,C,2,0,0,Mrs,2
2,1,3,female,-0.231523,0,0,-0.488580,S,1,0,1,Miss,1
3,1,1,female,0.448003,1,0,0.420494,S,2,0,0,Mrs,2
4,0,3,male,0.448003,0,0,-0.486064,S,1,0,1,Mr,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,-0.156020,0,0,-0.386454,S,1,0,1,Rev,1
887,1,1,female,-0.760043,0,0,-0.044356,S,1,0,1,Miss,1
888,0,3,female,-0.382528,1,2,-0.176164,S,4,0,0,Miss,4
889,1,1,male,-0.231523,0,0,-0.044356,C,1,0,1,Mr,1


In [91]:
data["Sex"] = data["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["Embarked","Pclass", "Title"], dtype=int)
test = pd.get_dummies(test, columns=["Embarked","Pclass", "Title"], dtype=int)

test = test.reindex(columns=data.drop("Survived", axis=1).columns, fill_value=0)

In [92]:
data

,Survived,Sex,Age,SibSp,Parch,Fare,family_size,is_child,is_alone,TicketGroupSize,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
0,0,0,-0.533534,1,0,-0.502163,2,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,1,1,0.674512,1,0,0.786404,2,0,0,2,...,0,0,0,0,0,1,0,0,0,0
2,1,1,-0.231523,0,0,-0.488580,1,0,1,1,...,0,1,0,0,0,0,0,0,0,0
3,1,1,0.448003,1,0,0.420494,2,0,0,2,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0.448003,0,0,-0.486064,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,0,-0.156020,0,0,-0.386454,1,0,1,1,...,0,0,0,0,0,0,0,1,0,0
887,1,1,-0.760043,0,0,-0.044356,1,0,1,1,...,0,1,0,0,0,0,0,0,0,0
888,0,1,-0.382528,1,2,-0.176164,4,0,0,4,...,0,1,0,0,0,0,0,0,0,0
889,1,0,-0.231523,0,0,-0.044356,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0


In [93]:
x_train = data.drop("Survived", axis = 1)
y_train = data["Survived"]

In [94]:
from sklearn.linear_model import LogisticRegression


clf = LogisticRegression(random_state=0, max_iter=1000 ).fit(x_train,y_train)
predictions = clf.predict(x_train)

from sklearn.metrics import accuracy_score

print(accuracy_score(y_train,predictions))

0.8327721661054994


In [95]:
submission_preds = clf.predict(test)

df = pd.DataFrame({"PassengerId":test_ids.values, "Survived" : submission_preds })

df.to_csv("submission.csv", index = False)